In [ ]:
# =====================================================================
#  티라노사우루스 발성 범위 추정 — 음원-필터 모델 + 몬테카를로 + Sobol 민감도
#  Google Colab에서 그대로 실행 가능 (런타임 > 모두 실행)
#  ※ 바꿀 값은 [1] 매개변수 칸과 [2] 관 실험 측정값 칸에만 있습니다.
# =====================================================================

# %% [0] 준비 (Colab 첫 실행 시 한 번만: 한글 글꼴·SALib 설치)
import sys, subprocess, os
if "google.colab" in sys.modules:
    subprocess.run("pip -q install SALib", shell=True)
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze

import glob
for f in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf"] + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
    if os.path.exists(f):
        fm.fontManager.addfont(f); plt.rcParams["font.family"] = fm.FontProperties(fname=f).get_name(); break
if plt.rcParams["font.family"] == ["sans-serif"]:
    for name in ["Noto Sans CJK KR", "Malgun Gothic", "AppleGothic"]:
        if any(name in x.name for x in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = name; break
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(2026)

# %% [1] 매개변수 — 범위와 근거 (보고서 표 1과 같은 값)
P = dict(
    M=(6000, 9500),          # 몸무게 kg, 균등분포 (Hutchinson 외 2011)
    kL=(0.75, 1.25),         # 막 길이 회귀식 L0=3.28*M^0.4 mm 의 외삽 보정 배수 (가정)
    a_mean=2.4, a_sd=2.9,    # 응력 상수 a kPa, 로그정규 (Riede 외 2015 표 1)
    b_mean=16.2, b_sd=3.1,   # 응력 지수 b, 정규 (Riede 외 2015 표 1)
    eps_min=0.03,            # 최소 늘임 (Riede 외 2015 시험 범위 하한)
    eps_max=(0.15, 0.35),    # 최대 늘임, 균등 (상한 0.35 = 시험 범위 상한, 하한은 가정)
    rho=1040.0,              # 조직 밀도 kg/m^3 (Titze 외 2016)
    c=(351, 356),            # 음속 m/s, 체온 36.3±2.5℃ (Flores 외 2026) + 습도 보정
    Lup=(1.2, 1.5),          # 후두→입 통로 m (Brochu 2003; Carr 2020)
    Lneck=(1.0, 1.4),        # 목 구간 기관 m (목뼈 치수로 추정, 가정 포함)
    Lchest=(0.1, 0.5),       # 가슴 안 기관 m (가정)
    phi=(0.001, 0.01),       # 닫힌 소리: 주머니 부피 / 몸 부피 (가정)
    rn=(0.02, 0.06),         # 닫힌 소리: 주머니 입구 반지름 m (가정)
    ln=(0.05, 0.20),         # 닫힌 소리: 주머니 입구 길이 m (가정)
    hear=(10, 3000),         # 걸러내기: 청각 범위 Hz (상한 Gleich 외 2005)
    allo_factor=4.0,         # 걸러내기: 몸무게 예측 최저 F0의 몇 배 범위까지 허용 (Riede 외 2019 그림 6)
    N=50_000,                # 조합마다 계산 횟수
)

# %% [2] 관 모형 실험 측정값 — 여기에 직접 잰 값을 넣으세요
# 관 길이(m), 관 안쪽 반지름(m), 실내 온도(℃), 측정한 첫째 공명 Hz (3회)
PIPE = pd.DataFrame([
    # 길이,  반지름, 온도, 측정1, 측정2, 측정3
    [0.300, 0.025, 20.0, None, None, None],
    [0.375, 0.025, 20.0, None, None, None],
    [0.575, 0.025, 20.0, None, None, None],
    [0.700, 0.025, 20.0, None, None, None],
    [0.850, 0.025, 20.0, None, None, None],
], columns=["L", "r", "T", "f1", "f2", "f3"])

def pipe_analysis(df):
    rows = []
    for _, d in df.iterrows():
        c = 331.3 + 0.606 * d["T"]
        f_plain = c / (4 * d["L"])
        f_corr = c / (4 * (d["L"] + 0.61 * d["r"]))
        for m in [d["f1"], d["f2"], d["f3"]]:
            if m is None or pd.isna(m): continue
            rows.append(dict(L=d["L"], 측정=m, 공식=f_plain, 끝보정공식=f_corr,
                             비율_공식=m / f_plain, 비율_끝보정=m / f_corr))
    return pd.DataFrame(rows)

pr = pipe_analysis(PIPE)
if len(pr):
    print("관 실험 결과\n", pr.round(3).to_string(index=False))
    print(f"측정/공식 비율 평균 {pr.비율_공식.mean():.3f} ± {pr.비율_공식.std():.3f}")
    print(f"측정/끝보정 비율 평균 {pr.비율_끝보정.mean():.3f} ± {pr.비율_끝보정.std():.3f}")
    KAPPA = (pr.비율_공식.mean(), pr.비율_공식.std())   # 통로 공식 오차 → 시뮬레이션에 반영
else:
    print("관 실험 값이 아직 없어 통로 공식 오차를 0으로 둡니다.")
    KAPPA = (1.0, 0.0)

# %% [3] 모델 — 음원(줄 모델) + 필터(관 공명 / 헬름홀츠 공명)
def lognorm_params(m, s):
    sl = np.sqrt(np.log(1 + (s / m) ** 2)); return np.log(m) - sl**2 / 2, sl

def f0(L0_m, a_kpa, b, eps, rho):
    sigma = a_kpa * 1e3 * np.exp(b * eps)            # 응력 Pa
    return 1 / (2 * L0_m * (1 + eps)) * np.sqrt(sigma / rho)

def tube_f1(c, L, kappa):          # 한쪽이 막힌 관: c / 4L
    return kappa * c / (4 * L)

def helmholtz(c, M, phi, rn, ln):  # 닫힌 소리: 주머니 공명
    V = phi * M / 1000.0; A = np.pi * rn**2; leff = ln + 1.7 * rn
    return c / (2 * np.pi) * np.sqrt(A / (V * leff))

def draw(n):
    u = lambda k: rng.uniform(*P[k], n)
    mu, sl = lognorm_params(P["a_mean"], P["a_sd"])
    return dict(M=u("M"), kL=u("kL"), a=rng.lognormal(mu, sl, n),
                b=np.clip(rng.normal(P["b_mean"], P["b_sd"], n), 1, None),
                eps_max=u("eps_max"), c=u("c"), Lup=u("Lup"), Lneck=u("Lneck"),
                Lchest=u("Lchest"), phi=u("phi"), rn=u("rn"), ln=u("ln"),
                kappa=rng.normal(KAPPA[0], KAPPA[1], n) if KAPPA[1] > 0 else np.full(n, KAPPA[0]))

CASES = ["후두+열린", "후두+닫힌", "울대+열린", "울대+닫힌"]

def run_case(case, x):
    L0 = 3.28e-3 * x["M"] ** 0.4 * x["kL"]     # m
    lo = f0(L0, x["a"], x["b"], P["eps_min"], P["rho"])
    hi = f0(L0, x["a"], x["b"], x["eps_max"], P["rho"])
    Lt = x["Lup"] if case.startswith("후두") else x["Lup"] + x["Lneck"] + x["Lchest"]
    if case.endswith("열린"):
        res = tube_f1(x["c"], Lt, x["kappa"])
    else:
        res = helmholtz(x["c"], x["M"], x["phi"], x["rn"], x["ln"])
    return lo, hi, res, Lt

# %% [4] 몬테카를로 + 걸러내기
summary, store = [], {}
for case in CASES:
    x = draw(P["N"])
    lo, hi, res, Lt = run_case(case, x)
    allo = 450 * x["M"] ** -0.38
    keep = (hi >= P["hear"][0]) & (lo <= P["hear"][1]) & (res <= P["hear"][1]) \
           & (lo <= allo * P["allo_factor"]) & (lo >= allo / P["allo_factor"])
    inter = (res >= lo * 0.8) & (res <= hi * 1.2)      # F0 범위와 공명이 겹침 → 음원-통로 상호작용 주의
    store[case] = dict(lo=lo[keep], hi=hi[keep], res=res[keep])
    q = lambda v: np.percentile(v[keep], [5, 50, 95])
    summary.append(dict(조합=case, 통과율=keep.mean(),
                        최저F0_5=q(lo)[0], 최저F0_50=q(lo)[1], 최저F0_95=q(lo)[2],
                        최고F0_5=q(hi)[0], 최고F0_50=q(hi)[1], 최고F0_95=q(hi)[2],
                        공명_5=q(res)[0], 공명_50=q(res)[1], 공명_95=q(res)[2],
                        상호작용비율=inter[keep].mean()))
S = pd.DataFrame(summary)
pd.set_option("display.width", 200)
print("\n[표 2] 네 조합의 범위 (Hz, 5–50–95 백분위)\n", S.round(2).to_string(index=False))
S.round(3).to_csv("result_ranges.csv", index=False, encoding="utf-8-sig")

# %% [5] 그림 1 — 네 조합의 범위 도표
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, r in S.iterrows():
    y = len(S) - i
    ax.plot([r.최저F0_5, r.최고F0_95], [y + 0.12] * 2, lw=6, color="#4C72B0", solid_capstyle="butt",
            label="소리 높이(F0) 범위" if i == 0 else None)
    ax.plot([r.공명_5, r.공명_95], [y - 0.12] * 2, lw=6, color="#DD8452", solid_capstyle="butt",
            label="울림(첫째 공명) 범위" if i == 0 else None)
ax.set_yticks(range(len(S), 0, -1), S.조합)
ax.set_xscale("log"); ax.set_xlabel("주파수 (Hz, 로그 눈금)")
ax.axvspan(1, 20, color="0.9", zorder=0); ax.text(1.5, 0.55, "초저주파(<20 Hz)", fontsize=8)
ax.legend(fontsize=8, loc="lower right"); ax.set_xlim(1, 3000)
plt.tight_layout(); plt.savefig("fig1_ranges.png", dpi=200)

# %% [6] 민감도 분석 (Sobol) — 어떤 미지수가 폭을 흔드는가
mu, sl = lognorm_params(P["a_mean"], P["a_sd"])
def problem(names, bounds, dists):
    return dict(num_vars=len(names), names=names, bounds=bounds, dists=dists)

prob_src = problem(["몸무게", "막길이 보정", "응력 a", "응력 지수 b", "최대 늘임"],
                   [list(P["M"]), list(P["kL"]), [mu, sl], [P["b_mean"], P["b_sd"]], list(P["eps_max"])],
                   ["unif", "unif", "lognorm", "norm", "unif"])
X = sobol_sample.sample(prob_src, 4096, seed=1)
L0 = 3.28e-3 * X[:, 0] ** 0.4 * X[:, 1]
b = np.clip(X[:, 3], 1, None)
Y_lo = np.log10(f0(L0, X[:, 2], b, P["eps_min"], P["rho"]))
Y_hi = np.log10(f0(L0, X[:, 2], b, X[:, 4], P["rho"]))
Si_lo = sobol_analyze.analyze(prob_src, Y_lo, seed=1)
Si_hi = sobol_analyze.analyze(prob_src, Y_hi, seed=1)

prob_open = problem(["음속", "후두→입", "목 구간", "가슴 구간"],
                    [list(P["c"]), list(P["Lup"]), list(P["Lneck"]), list(P["Lchest"])], ["unif"] * 4)
Xo = sobol_sample.sample(prob_open, 4096, seed=1)
Si_open = sobol_analyze.analyze(prob_open, Xo[:, 0] / (4 * Xo[:, 1:].sum(axis=1)), seed=1)

prob_cl = problem(["음속", "몸무게", "주머니 부피비", "입구 반지름", "입구 길이"],
                  [list(P["c"]), list(P["M"]), list(P["phi"]), list(P["rn"]), list(P["ln"])], ["unif"] * 5)
Xc = sobol_sample.sample(prob_cl, 4096, seed=1)
Si_cl = sobol_analyze.analyze(prob_cl, helmholtz(*Xc.T), seed=1)

rows = []
for tag, prob, Si in [("최저 F0", prob_src, Si_lo), ("최고 F0", prob_src, Si_hi),
                      ("울대+열린 공명", prob_open, Si_open), ("닫힌 소리 공명", prob_cl, Si_cl)]:
    for n, s1, st in zip(prob["names"], Si["S1"], Si["ST"]):
        rows.append(dict(출력=tag, 변수=n, S1=s1, ST=st))
SS = pd.DataFrame(rows)
print("\n[표 3] Sobol 민감도 지수 (ST가 클수록 그 변수가 폭을 많이 흔듦)\n",
      SS.sort_values(["출력", "ST"], ascending=[True, False]).round(3).to_string(index=False))
SS.round(3).to_csv("result_sobol.csv", index=False, encoding="utf-8-sig")

# %% [7] 그림 2 — 민감도 막대그래프
outs = ["최저 F0", "최고 F0", "울대+열린 공명", "닫힌 소리 공명"]
fig, axs = plt.subplots(1, 4, figsize=(11, 3.2), sharex=True)
for ax, o in zip(axs, outs):
    d = SS[SS.출력 == o].sort_values("ST")
    ax.barh(d.변수, d.ST.clip(lower=0), color="#55A868"); ax.set_title(o, fontsize=10)
    ax.set_xlim(0, 1); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig("fig2_sobol.png", dpi=200)
print("\n저장: result_ranges.csv, result_sobol.csv, fig1_ranges.png, fig2_sobol.png")
